In [0]:
%python

import sys, os
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
root_folder = "/".join(notebook_path.split("/")[:-2])

sys.path.append('/Workspace/' + root_folder + '/Functions')

from helper_functions import apply_schema
from schema_definitions import silver_schema, gold_schema
from pyspark.sql.functions import col, left, current_timestamp, to_date, min, max, avg, sum, stddev, lit

In [0]:
@dlt.table(
    name=f"wind_turbines_raw",
    comment="table to store raw bronze data",
    partition_cols=["turbine_id"]
)
@dlt.expect_or_fail("valid_wind_direction", "wind_direction BETWEEN 0 AND 359")
def wind_turbines_raw():
    # read raw data and adds metadata columns
    new_bronze_df = (
                spark.readStream.format("cloudFiles")
                .option("cloudFiles.format", "csv")
                .option("delimiter", ",")
                .option("inferSchema", False)
                .load("/mnt/rawdata/Test/Turbines/*/")
                .withColumn("input_file_name", col("_metadata.file_path"))
                .withColumn("import_timestamp", current_timestamp())
                )
    return new_bronze_df

In [0]:
@dlt.table(
    name=f"wind_turbines_cleansed",
    comment=f"Table to store cleansed silver layer",
    partition_cols=["turbine_id", "date"]
)
def wind_turbines_cleansed():
    bronze_df = spark.read.table("wind_turbines_raw")
    timestamp = bronze_df.select("timestamp").distinct()
    turbine = bronze_df.select("turbine_id").distinct()
    cross_df = timestamp.crossJoin(turbine)

    missing_df = (
                cross_df.alias("c")
                        .join(bronze_df.alias("b"), ["timestamp", "turbine_id"], "left_anti")
                        .select(*[col(f"c.{x}") if x in cross_df.columns else (lit(current_timestamp()).alias(x) if x == "import_timestamp" else lit(None).cast(bronze_df.schema[x].dataType).alias(x)) for x in bronze_df.columns])
                )

    silver_df = (
                bronze_df.union(missing_df)
                        .withColumn("date", to_date(left("timestamp", lit(10)), "yyyy-MM-dd"))
                        .withColumn("timestamp", col("timestamp").cast("timestamp"))
                )

    # Calculates min, max, sum for power_output
    summary_df = (
                silver_df
                    # .withWatermark("timestamp", "10 minutes")
                    # .groupBy(window(silver_df.timestamp, "10 minutes", "5 minutes"), "date", "turbine_id")
                    .groupBy("date", "turbine_id")
                    .agg(min("power_output").alias("min_power_output"),
                        max("power_output").alias("max_power_output"),
                        avg("power_output").alias("avg_power_output"),
                        sum("power_output").alias("sum_power_output"),
                        stddev("power_output").alias("stddev_power_output"))
                )

    # Joins anomaly_df to gold_df
    silver_df = (
                silver_df.alias("i")
                        .join(summary_df.alias("u"), ["date", "turbine_id"])
                        .select("i.*", "u.avg_power_output", "u.stddev_power_output")
            )

    # Identifies anomaly
    silver_df = silver_df.withColumn("anomaly", (col("power_output") < col("avg_power_output") - col("stddev_power_output") * lit(2)) | (col("power_output") > col("avg_power_output") + col("stddev_power_output") * lit(2)))

    silver_df = apply_schema(silver_df, silver_schema, True)
    
    return silver_df

In [0]:
@dlt.table(
    name=f"wind_turbines_aggregated",
    comment=f"Table to store aggregated gold layer",
    partition_cols=["turbine_id", "date"]
)
def wind_turbines_aggregated():
    silver_df = spark.read.table("wind_turbines_cleansed")
    gold_df = (
                silver_df
                    .filter((col("anomaly") == False) | (col("input_file_name") != "UNDEF"))
                    .groupBy("date", "turbine_id")
                    .agg(min("power_output").alias("min_power_output"),
                        max("power_output").alias("max_power_output"),
                        avg("power_output").alias("avg_power_output"))
                )

    # Apply schema
    gold_df = apply_schema(gold_df, gold_schema, False)

    return gold_df